# DAY 2: Data Pre-processing

Rewrite raw Amazon product descriptions into a clean, standardized format using an LLM via OpenRouter.

**Pipeline steps:**
1. Load raw items from Hugging Face Hub
2. Assign stable ids to items
3. Rewrite each product description via OpenRouter (with checkpointing)
4. Split and push processed dataset back to Hugging Face Hub

In [87]:
import os
import time
import pickle
from pathlib import Path
from typing import Optional

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

from pricer.items import Item

load_dotenv(override=True)

True

In [88]:
def create_openrouter_client() -> OpenAI:
    """Create an OpenAI-compatible client configured for OpenRouter."""
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise ValueError("OPENROUTER_API_KEY is not set in environment")
    return OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")

In [89]:
def load_items_from_hub(username: str, lite_mode: bool) -> list[Item]:
    """Load raw items dataset from Hugging Face Hub and return combined splits."""
    dataset_name = f"{username}/items_raw_curated" if lite_mode else f"{username}/items_raw_full"
    train, val, test = Item.from_hub(dataset_name)
    items = train + val + test
    print(f"Loaded {len(items):,} items from {dataset_name}")
    return items

In [90]:
def assign_item_ids(items: list[Item]) -> None:
    """Assign stable numeric ids so each rewritten summary maps to the correct item."""
    for index, item in enumerate(items):
        item.id = index

In [91]:
def get_system_prompt() -> str:
    """Return the instruction prompt for standardizing product text."""
    return """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [92]:
def rewrite_product_summary(
    client: OpenAI,
    model_name: str,
    system_prompt: str,
    product_text: str,
    max_retries: int = 3,
) -> str:
    """Rewrite one product description into a standardized summary format."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": product_text},
                ],
                temperature=0.2,
            )
            return response.choices[0].message.content.strip()
        except Exception:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 * (attempt + 1))
    return ""

In [93]:
def resolve_checkpoint_path(checkpoint_file: str) -> Path:
    """Resolve checkpoint path and ensure its parent directory exists."""
    checkpoint_path = Path(checkpoint_file)
    if not checkpoint_path.is_absolute():
        checkpoint_path = (Path.cwd() / checkpoint_path).resolve()
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    return checkpoint_path


def save_checkpoint(items: list[Item], checkpoint_path: Path) -> None:
    """Persist current processing state to disk for resume support."""
    with checkpoint_path.open("wb") as file_obj:
        pickle.dump(items, file_obj)


def load_checkpoint(checkpoint_path: Path) -> Optional[list[Item]]:
    """Load previously saved processing state if available."""
    if not checkpoint_path.exists():
        return None
    with checkpoint_path.open("rb") as file_obj:
        return pickle.load(file_obj)

In [94]:
def process_items_with_openrouter(
    items: list[Item],
    client: OpenAI,
    model_name: str,
    checkpoint_path: Path,
    checkpoint_every: int = 100,
    pause_seconds: float = 0.0,
) -> None:
    """Generate summaries for all items, resuming previously completed items and checkpointing progress."""
    system_prompt = get_system_prompt()
    completed = 0

    for item in tqdm(items):
        if item.summary:
            completed += 1
            continue

        item.summary = rewrite_product_summary(
            client=client,
            model_name=model_name,
            system_prompt=system_prompt,
            product_text=item.full,
        )
        completed += 1

        if completed % checkpoint_every == 0:
            save_checkpoint(items, checkpoint_path)
            print(f"Checkpoint saved at {completed:,} items")

        if pause_seconds > 0:
            time.sleep(pause_seconds)

    save_checkpoint(items, checkpoint_path)
    print("Processing complete and final checkpoint saved")

In [95]:
def clear_unneeded_fields(items: list[Item]) -> None:
    """Remove fields not needed in the final processed dataset."""
    for item in items:
        item.full = None
        item.id = None


def split_processed_items(items: list[Item], lite_mode: bool) -> dict:
    """Create train/validation/test splits matching the Day 2 convention."""
    if lite_mode:
        train = items[:20_000]
        val = items[20_000:21_000]
        test = items[21_000:]
        return {"curated": (train, val, test)}

    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    return {
        "full": (train, val, test),
        "lite": (train_lite, val_lite, test_lite),
    }


def push_processed_datasets(
    username: str,
    splits: dict[str, tuple[list[Item], list[Item], list[Item]]],
) -> None:
    """Push processed dataset splits to Hugging Face Hub."""
    for variant, (train, val, test) in splits.items():
        dataset_name = f"{username}/items_{variant}"
        Item.push_to_hub(dataset_name, train, val, test)
        print(
            f"Pushed {dataset_name}: "
            f"train={len(train):,}, validation={len(val):,}, test={len(test):,}"
        )

In [96]:
def run_pipeline(
    username: str,
    lite_mode: bool = True,
    model_name: str = "openai/gpt-4o-mini",
    checkpoint_file: str = "checkpoint_file.pkl",
) -> None:
    """Run the complete Day 2 preprocessing pipeline using OpenRouter."""
    checkpoint_path = resolve_checkpoint_path(checkpoint_file)

    items = load_checkpoint(checkpoint_path)
    if items is None:
        items = load_items_from_hub(username=username, lite_mode=lite_mode)
        assign_item_ids(items)
        save_checkpoint(items, checkpoint_path)
        print(f"Initialized new run. Checkpoint saved at {checkpoint_path}")
    else:
        print(f"Resumed from checkpoint with {len(items):,} items at {checkpoint_path}")

    client = create_openrouter_client()
    process_items_with_openrouter(
        items=items,
        client=client,
        model_name=model_name,
        checkpoint_path=checkpoint_path,
        checkpoint_every=100,
        pause_seconds=0.0,
    )

    clear_unneeded_fields(items)
    splits = split_processed_items(items, lite_mode=lite_mode)
    push_processed_datasets(username=username, splits=splits)
    print("Day 2 pipeline completed")

## Run the pipeline

Set `username` to your Hugging Face username and run the cell below.

- `lite_mode=True` uses your `items_raw_curated` dataset (fast, cheap)
- `lite_mode=False` uses `items_raw_full` (full dataset, ~$30 cost)

In [ ]:
run_pipeline(
    username="mainanorbert",
    lite_mode=True,
    model_name="openai/gpt-4o-mini",
    checkpoint_file="checkpoint_file.pkl",
)

Resumed from checkpoint with 10,001 items at /home/nober/andela/ai_engineering/week6/checkpoint_file.pkl


Checkpoint saved at 100 items


Checkpoint saved at 200 items


Checkpoint saved at 300 items


Checkpoint saved at 400 items


...

Checkpoint saved at 5,400 items


Checkpoint saved at 5,500 items


Checkpoint saved at 5,600 items


Checkpoint saved at 5,700 items


Checkpoint saved at 5,800 items


## Push already-processed items

Use this cell to push whatever has been summarized so far from the checkpoint, without waiting for the full pipeline to finish.

In [ ]:
def push_partial_results(
    username: str,
    checkpoint_file: str = "checkpoint_file.pkl",
    dataset_suffix: str = "items_curated",
) -> None:
    """Load checkpoint, push only completed (summarized) items to Hugging Face Hub."""
    checkpoint_path = resolve_checkpoint_path(checkpoint_file)

    if not checkpoint_path.exists():
        print(f"No checkpoint found at {checkpoint_path}")
        return

    with checkpoint_path.open("rb") as file_obj:
        items = pickle.load(file_obj)

    completed = [item for item in items if item.summary is not None]
    print(f"{len(completed):,} summarized items out of {len(items):,} total")

    if not completed:
        print("No summarized items yet. Nothing to push.")
        return

    for item in completed:
        item.full = None
        item.id = None

    total = len(completed)
    train_end = int(total * 0.90)
    val_end = int(total * 0.95)

    train = completed[:train_end]
    val = completed[train_end:val_end]
    test = completed[val_end:]

    dataset_name = f"{username}/{dataset_suffix}"
    Item.push_to_hub(dataset_name, train, val, test)
    print(
        f"Pushed {dataset_name}: "
        f"train={len(train):,}, validation={len(val):,}, test={len(test):,}"
    )


push_partial_results(
    username="mainanorbert",
    checkpoint_file="checkpoint_file.pkl",
    dataset_suffix="items_curated",
)

In [100]:
from datasets import load_dataset

ds = load_dataset("mainanorbert/items_curated")

print(ds["train"][0])

{'title': 'Romalon 990-53 Humidifier Solenoid Vavle Replacement Fit for Gen-eral & Gene-ralaire Humidifier,24 Volts Replaces 1042 1042LH 1137 1042L', 'category': 'Appliances', 'price': 19.99, 'full': None, 'weight': 0.4625, 'summary': 'Title: Universal Humidifier Solenoid Valve Replacement  \nCategory: Home Appliances  \nBrand: Romalon  \nDescription: This 24-volt solenoid valve is designed to replace various humidifier models for efficient moisture control.  \nDetails: It features easy installation and is made from durable materials, ensuring compatibility with a wide range of humidifiers.', 'prompt': None, 'id': None}
